In [1]:
# ignore the unnecessary warning messages
import warnings
warnings.filterwarnings("ignore")

In [2]:
import json

with open("document_chunks_and_questions.json", "r", encoding="utf-8") as file:
    data = json.load(file)

examples_chunks_list = data['examples_chunks_list']
user_questions = data['user_questions']

In [3]:
import chromadb

# Initialize ChromaDB (RAM only)
# makes connections with chromaDB
client = chromadb.Client()

# ChromaDB (Persistent Storage - SSD)
# client = chromadb.PersistentClient(
#     path="./vector_db"
# )

# Create collection
# A single RAM only or ChromaDB folder (vector_db) can contain multiple collections. There is no requirement to create a separate folder for each collection.
collection = client.get_or_create_collection(
    name="my_documents",
    metadata={"hnsw:space": "cosine"} # using the built in cosine similarity method
)

# Create second collection
# collection2 = client.get_or_create_collection(
#     name="research_documents",
#     metadata={"hnsw:space": "l2"} // using Euclidean Distance (L2 Distance) method
# )

In [4]:
# creating the client - make the connections with ollama api endpoints
import ollama

ollama_client = ollama.Client(
    host="http://127.0.0.1:11434"
)

print("Ollama Client connection is successful")

Ollama Client connection is successful


In [5]:
def embedding_model(content):
    return ollama_client.embed(
        model="nomic-embed-text:latest",
        input=content
    )

chunk_embedding = embedding_model(examples_chunks_list)['embeddings']
questions_embedding = embedding_model(user_questions)['embeddings']

In [6]:
collection.add(
    ids=["doc1", "doc2", "doc3", "doc4"],
    documents=examples_chunks_list,
    embeddings=chunk_embedding
)

print("Documents successfully stored!")

Documents successfully stored!


In [ ]:
# Search VectorDB
for question, q_embedding in zip(user_questions, questions_embedding):
    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=1
    )

    # Display results
    print(f"Question: {question}")
    # loop add because if n_result > 1
    for doc, distance in zip(
        results["documents"][0],
        results["distances"][0]
    ):
        print("Document:", doc)
        print("Distance:", distance)
        print("----------------")

Question: What is Python and what is it used for?
Document: Python is a high-level programming language. It is widely used for web development, data science, machine learning, automation, and scripting.
Distance: 0.14378798007965088
----------------
Question: How does Retrieval-Augmented Generation (RAG) work?
Document:  RAG stands for Retrieval-Augmented Generation. RAG allows an LLM to retrieve information from an external knowledge source before generating an answer.
Distance: 0.19611215591430664
----------------
Question: What are embeddings and how do vector databases use them?
Document:  Vector databases store numerical representations called embeddings. They allow applications to search for semantically similar pieces of information.
Distance: 0.1542101502418518
----------------


# ChromaDB – In-Memory vs Persistent Storage

## 1. Introduction

ChromaDB is a vector database used to store embeddings and perform similarity searches in Retrieval-Augmented Generation (RAG) applications.

ChromaDB supports two storage approaches:

1. **In-Memory Storage** – Stores vector data temporarily in memory (RAM).
2. **Persistent Storage** – Stores vector data on disk for future use.

---

## 2. In-Memory Storage

In-memory storage keeps the VectorDB data in RAM instead of saving it to a permanent location.

### Implementation

```python
import chromadb

# Initialize ChromaDB (In-Memory)
client = chromadb.Client()

# Create collection
collection = client.get_or_create_collection(
    name="my_documents",
    metadata={"hnsw:space": "cosine"}
)
```

### Explanation

- `chromadb.Client()` initializes an in-memory ChromaDB instance.
- Data is stored temporarily rather than in a user-specified persistent directory.
- Collections can be created to organize documents and embeddings.
- Data is not retained after the in-memory database is shut down.

### Advantages

- Simple to implement.
- No need to manage database files.
- Useful for testing and experimentation.
- Convenient for temporary vector storage.

### Disadvantages

- Data is not retained after the database is shut down.
- Embeddings and documents must be generated or loaded again.
- Not suitable when persistent storage is required.

---

## 3. Persistent Storage

Persistent storage saves vector database information to disk, allowing it to be reused across application restarts.

### Implementation

```python
import chromadb

# Initialize ChromaDB (Persistent Storage)
client = chromadb.PersistentClient(
    path="./vector_db"
)

# Create collection
collection = client.get_or_create_collection(
    name="my_documents",
    metadata={"hnsw:space": "cosine"}
)
```

### Explanation

- `chromadb.PersistentClient()` initializes a persistent ChromaDB instance.
- `path="./vector_db"` specifies the directory where database files are stored.
- Collections and their stored data are retained on disk.
- Previously stored embeddings and documents can be retrieved when the application restarts.

### Advantages

- Data is retained after application restarts.
- Avoids repeatedly generating and storing embeddings.
- Suitable for long-running RAG applications.
- Supports reuse of previously stored documents and embeddings.

### Disadvantages

- Requires disk storage.
- Database files need to be managed.
- Requires additional storage management compared to temporary in-memory usage.

---

## 4. Comparison

| Feature | In-Memory | Persistent |
|---|---|---|
| Initialization | `chromadb.Client()` | `chromadb.PersistentClient()` |
| Storage | RAM-backed | Disk-backed |
| Data persistence | Temporary | Permanent until deleted |
| Application restart | Data is not retained | Data is retained |
| Storage directory | Not specified | Required |
| Best use case | Testing and experimentation | Production and reusable applications |

---

## 5. When to Use Which?

### Use In-Memory Storage When:
- Testing ChromaDB functionality.
- Experimenting with embeddings.
- Performing temporary similarity searches.
- Building prototypes.

### Use Persistent Storage When:
- Developing a RAG application.
- Working with large document collections.
- Reusing previously generated embeddings.
- Avoiding repeated document ingestion and embedding generation.

---

## 6. Conclusion

Both approaches provide vector storage and similarity search functionality.

- **In-Memory ChromaDB:** Suitable for temporary storage, testing, and experimentation.
- **Persistent ChromaDB:** Suitable for applications that require data retention and reuse.

For a RAG application, persistent storage is generally recommended because it allows previously generated embeddings and documents to be reused without repeating the entire ingestion process.

# Similarity Scoring Methods in Python for RAG

## 1. Introduction

Similarity scoring methods are used to measure how closely two vectors or data objects are related.

In Retrieval-Augmented Generation (RAG), these methods help identify and retrieve documents that are most relevant to a user's query.

Text is converted into numerical embeddings, and similarity scoring methods compare these embeddings to find relevant information.

## 2. Types of Similarity Scoring Methods

### 2.1 Cosine Similarity

Cosine Similarity measures the cosine of the angle between two vectors.

**Python Implementation:**

```python
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

vector_a = np.array([1, 2, 3])
vector_b = np.array([2, 3, 4])

score = cosine_similarity(vector_a, vector_b)

print("Cosine Similarity:", score)
```

**Characteristics:**
- Score ranges from -1 to +1.
- +1 indicates the same direction.
- 0 indicates perpendicular vectors.
- -1 indicates opposite directions.
- Higher score indicates greater similarity.

**Use case:** Semantic text search and document retrieval.

### 2.2 Euclidean Distance (L2 Distance)

Euclidean Distance measures the straight-line distance between two vectors.

**Python Implementation:**

```python
def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

score = euclidean_distance(vector_a, vector_b)

print("Euclidean Distance:", score)
```

**Characteristics:**
- Range: 0 to infinity.
- 0 indicates identical vectors.
- Lower distance indicates greater similarity.
- Sensitive to vector magnitude.

**Use case:** Geometric distance comparison between embeddings.

### 2.3 Manhattan Distance (L1 Distance)

Manhattan Distance measures the sum of absolute differences between corresponding vector elements.

**Python Implementation:**

```python
def manhattan_distance(a, b):
    return np.sum(np.abs(a - b))

score = manhattan_distance(vector_a, vector_b)

print("Manhattan Distance:", score)
```

**Characteristics:**
- Range: 0 to infinity.
- 0 indicates identical vectors.
- Lower distance indicates greater similarity.
- Measures absolute differences across dimensions.

**Use case:** Certain numerical and high-dimensional data comparisons.

### 2.4 Dot Product Similarity

Dot Product measures the sum of the products of corresponding vector elements.

**Python Implementation:**

```python
def dot_product_similarity(a, b):
    return np.dot(a, b)

score = dot_product_similarity(vector_a, vector_b)

print("Dot Product:", score)
```

**Characteristics:**
- Higher score generally indicates greater similarity.
- Considers both vector direction and magnitude.
- Magnitude can influence the score.

**Use case:** Embedding retrieval, especially when using compatible normalized or model-trained embeddings.

### 2.5 Jaccard Similarity

Jaccard Similarity measures the similarity between two sets based on their intersection and union.

**Python Implementation:**

```python
def jaccard_similarity(a, b):
    intersection = len(set(a) & set(b))
    union = len(set(a) | set(b))

    return intersection / union if union else 1.0

set_a = ["RAG", "AI", "Python", "VectorDB"]
set_b = ["RAG", "AI", "Machine Learning"]

score = jaccard_similarity(set_a, set_b)

print("Jaccard Similarity:", score)
```

**Characteristics:**
- Score ranges from 0 to 1.
- 1 indicates identical sets.
- 0 indicates no common elements.
- Compares shared elements rather than vector geometry.

**Use case:** Keyword matching, token comparison, and document set comparison.

---

## 3. Comparison of Similarity Methods

| Method | Comparison Type | Best Result |
|---|---|---|
| Cosine Similarity | Vector angle | Higher |
| Euclidean Distance | Straight-line distance | Lower |
| Manhattan Distance | Absolute differences | Lower |
| Dot Product | Vector multiplication | Higher |
| Jaccard Similarity | Set intersection and union | Higher |

## 4. ChromaDB Similarity Scoring

ChromaDB supports three distance metrics for its HNSW index.

### 4.1 Cosine Distance

Configuration:

```python
metadata={"hnsw:space": "cosine"}
```

Lower distance indicates greater similarity.

### 4.2 Euclidean Distance (L2)

Configuration:

```python
metadata={"hnsw:space": "l2"}
```

Measures squared Euclidean distance between vectors.

Lower distance indicates greater similarity.

### 4.3 Inner Product

Configuration:

```python
metadata={"hnsw:space": "ip"}
```

Measures inner product-based distance.

Lower distance indicates greater similarity.

**Important:** ChromaDB returns distances rather than similarity scores. Its inner product distance is not simply the raw dot product.

## 5. ChromaDB Implementation

```python
import chromadb

# Initialize in-memory ChromaDB
client = chromadb.Client()

# Create collection using cosine distance
collection = client.get_or_create_collection(
    name="my_documents",
    metadata={"hnsw:space": "cosine"}
)

# Add documents and embeddings
collection.add(
    ids=["1", "2"],
    documents=[
        "RAG stands for Retrieval-Augmented Generation.",
        "Python is a programming language."
    ],
    embeddings=[
        [0.1, 0.2, 0.3],
        [0.4, 0.5, 0.6]
    ]
)

# Perform similarity search
results = collection.query(
    query_embeddings=[[0.1, 0.2, 0.3]],
    n_results=2
)

print(results)
```

**Note:** The example embeddings are illustrative. In a real RAG application, generate embeddings using the same embedding model for both stored documents and query text.

## 6. Important Differences

| Feature | Similarity Score | Distance Score |
|---|---|---|
| Meaning | Measures closeness | Measures separation |
| Higher value | Generally more similar | Generally less similar |
| Lower value | Generally less similar | Generally more similar |
| Examples | Cosine similarity, dot product | Euclidean, Manhattan |

## 7. Which Method Should Be Used in RAG?

| Scenario | Recommended Method |
|---|---|
| Semantic document retrieval | Cosine Similarity |
| Normalized embeddings | Cosine or Dot Product |
| Magnitude-sensitive retrieval | Dot Product |
| Geometric distance comparison | Euclidean Distance |
| Keyword or token matching | Jaccard Similarity |

## 8. Conclusion

Similarity scoring methods play an important role in identifying relevant documents in RAG applications.

- Cosine Similarity compares the direction of vectors.
- Euclidean Distance measures straight-line distance.
- Manhattan Distance measures absolute differences.
- Dot Product measures vector multiplication.
- Jaccard Similarity compares common elements between sets.

For a ChromaDB-based RAG application, Cosine Similarity is a good starting choice for semantic document retrieval.

ChromaDB supports Cosine, L2, and Inner Product distance metrics through its HNSW index.